# BEATs frozen downstream benchmark

Objective: compare balanced logistic regression, a one-hidden-layer MLP, and a two-hidden-layer MLP on frozen 768-d mean-pooled BEATs embeddings. Inputs come from the prior audited extraction archive and are aligned to the 6,898-row ICBHI cycle manifest by unique `cycle_id`. Test uses the official challenge split; patients 156 and 218 cross train/test, so this is not patient-independent. Validation is patient-grouped and drawn only from official train. Outputs go to `result/icbhi_frozen_downstream/architecture/beats/`. This is a formal local controlled baseline, not a BEATs-paper reproduction.

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'baseline').exists(): ROOT = ROOT.parent
os.chdir(ROOT)
print('repo:', ROOT)
print('python:', sys.executable)

Run `mode='smoke'` for a bounded wiring check, or `mode='full'` for the fixed three-seed formal protocol. The shared runner writes config, environment receipt, feature checksum/provenance, validation assignment, metrics, predictions, confusion matrices, curves, checkpoints, and logs.

In [ ]:
from baseline.common.run import run_backbone, consolidate_full_results
mode = 'full'
rows = run_backbone('beats', mode=mode)
if mode == 'full': consolidate_full_results()
len(rows)

## Flat4 imbalance-aware loss comparison

This controlled extension fixes the MLP2 architecture and all formal-v1 training settings, and varies only the loss: unweighted CE, balanced class-weighted CE, focal loss (gamma=2, no alpha), or effective-number class-balanced CE (beta=0.9999). Outputs are isolated under `result/icbhi_frozen_downstream/loss/beats/`.

In [ ]:
from baseline.common.imbalance import run_imbalance_backbone, consolidate_imbalance_results
imbalance_mode = 'full'
imbalance_rows = run_imbalance_backbone('beats', mode=imbalance_mode)
if imbalance_mode == 'full': consolidate_imbalance_results()
len(imbalance_rows)

## Strict patient-grouped long-tail policy benchmark v3

This separate protocol builds fixed nested patient-grouped folds over all 6,898 cycles and compares six standard policies on the same frozen BEATs representation and MLP2 head. Outer test folds never select epochs or policy settings. Logit Adjustment, LDAM-DRW, and cRT use pre-registered local adaptations recorded in `result/icbhi_strict_patient_long_tail/comparison/protocol.json`. This is a local controlled benchmark, not an official method-paper reproduction.

In [ ]:
from baseline.common.strict_patient_v3 import run_strict_patient_v3
strict_mode = 'smoke'
strict_rows = run_strict_patient_v3(mode=strict_mode)
len(strict_rows)